# 03-03 LLM API 调用实战

**本节目标**：
- 掌握 OpenAI / Anthropic / 通义千问 三大 API 调用方式
- 理解 Token 计费和成本控制
- 实战异步并发调用和流式输出
- 了解 API 调用的错误处理和重试

---

In [ ]:
import os, sys, asyncio, time
sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv("../.env")

from utils.llm_client import call_llm, call_llm_async

print("API Keys 配置状态:")
for key in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "DASHSCOPE_API_KEY"]:
    status = "✅ 已配置" if os.environ.get(key) else "⬜ 未配置"
    print(f"  {key}: {status}")

## 1. 统一封装调用（推荐）

`utils/llm_client.py` 封装了三个提供商的 API，统一接口，方便切换模型。

In [ ]:
# 使用统一封装
prompt = "用一句话描述B站广告CTR优化的核心思路"

# 尝试调用（如果没有 API key 会返回 mock 回复）
try:
    response = call_llm(
        prompt=prompt,
        provider="openai",       # 可切换: "anthropic", "dashscope"
        model="gpt-4o-mini",     # 成本最低的 GPT-4 级别模型
        system="你是B站商业化数据分析专家",
        temperature=0.7,
        max_tokens=200,
    )
    print(f"回复: {response}")
except Exception as e:
    print(f"调用失败（需要配置 API Key）: {e}")
    print("Mock 回复: 通过提升广告相关性和用户匹配精度，同时优化创意质量，可有效提升CTR。")

## 2. OpenAI API 直接调用

In [ ]:
try:
    from openai import OpenAI
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "sk-placeholder"))
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "你是B站广告审核专家，输出JSON格式"},
            {"role": "user", "content": "审核这个广告标题：'限时折扣！全场5折！最便宜！第一！'"}
        ],
        temperature=0.3,
        max_tokens=300,
        response_format={"type": "json_object"},  # 强制 JSON 输出
    )
    
    content = response.choices[0].message.content
    usage = response.usage
    
    print(f"回复: {content}")
    print(f"\nToken 用量:")
    print(f"  输入: {usage.prompt_tokens} tokens")
    print(f"  输出: {usage.completion_tokens} tokens")
    print(f"  总计: {usage.total_tokens} tokens")
    
    # GPT-4o-mini 定价: $0.15/1M input, $0.60/1M output
    cost = usage.prompt_tokens * 0.15e-6 + usage.completion_tokens * 0.60e-6
    print(f"  本次费用: ${cost:.6f} (约 ¥{cost*7.2:.5f})")
    
except Exception as e:
    print(f"OpenAI 调用示例（需配置 OPENAI_API_KEY）")
    print("""
关键参数说明：
  model: 模型选择（成本从低到高）
    gpt-4o-mini    $0.15/1M in, $0.60/1M out   → 日常任务首选
    gpt-4o         $2.50/1M in, $10.0/1M out   → 复杂推理任务
    
  temperature: 创造性（0=确定性高, 1=创造性强）
    分类/提取任务: 0-0.3
    文案生成任务: 0.7-1.0
    
  response_format: {"type": "json_object"} 强制 JSON 格式
  max_tokens: 限制输出长度，控制成本
    """)

## 3. Anthropic (Claude) API

In [ ]:
try:
    import anthropic
    client_anthropic = anthropic.Anthropic(
        api_key=os.environ.get("ANTHROPIC_API_KEY", "placeholder")
    )
    
    message = client_anthropic.messages.create(
        model="claude-haiku-4-5-20251001",  # 最快最便宜
        max_tokens=300,
        system="你是B站广告文案创作专家",
        messages=[
            {"role": "user", "content": "为一款游戏皮肤写一个吸引眼球的广告标题，15字以内"}
        ]
    )
    print(f"Claude 回复: {message.content[0].text}")
    print(f"用量: 输入 {message.usage.input_tokens}, 输出 {message.usage.output_tokens}")
    
except Exception as e:
    print("Anthropic Claude API（需配置 ANTHROPIC_API_KEY）")
    print("""
Anthropic vs OpenAI API 差异：
  1. 消息格式：Anthropic 把 system 和 messages 分开传，OpenAI 放在 messages 列表里
  2. 模型命名：claude-haiku / sonnet / opus（速度↑，能力↑，价格↑）
  3. 工具调用：Anthropic 用 tools 参数，格式类似但字段名不同
  4. 流式输出：Anthropic 用 with client.messages.stream() 上下文管理器
    """)

## 4. 通义千问（DashScope）API

In [ ]:
# DashScope 兼容 OpenAI API 格式，只需修改 base_url
try:
    from openai import OpenAI as DashScopeClient
    
    ds_client = DashScopeClient(
        api_key=os.environ.get("DASHSCOPE_API_KEY", "placeholder"),
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    )
    
    response = ds_client.chat.completions.create(
        model="qwen-turbo",
        messages=[
            {"role": "system", "content": "你是B站广告数据分析师"},
            {"role": "user", "content": "如果CTR从2%降到1.5%，会有什么影响？"}
        ],
        max_tokens=200,
    )
    print(f"通义千问回复: {response.choices[0].message.content}")
    
except Exception as e:
    print("通义千问 API（需配置 DASHSCOPE_API_KEY）")
    print("""
通义千问模型选型：
  qwen-turbo    → 速度最快，成本最低，日常任务
  qwen-plus     → 平衡性能与成本
  qwen-max      → 最强能力，复杂推理
  qwen-long     → 超长上下文（1M tokens）
  
优势：
  - 国内网络访问稳定（无需翻墙）
  - 中文理解能力强
  - 有免费额度（适合学习）
  - 兼容 OpenAI SDK，迁移成本低
    """)

## 5. 流式输出（Streaming）

In [ ]:
def stream_demo():
    """演示流式输出 - 适合长文本生成场景（用户体验更好）"""
    try:
        from openai import OpenAI
        client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "placeholder"))
        
        print("流式输出（token by token）：", end="", flush=True)
        
        with client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": "用3句话介绍B站广告投放平台"}],
            stream=True,
        ) as stream:
            full_text = ""
            for chunk in stream:
                if chunk.choices[0].delta.content:
                    token = chunk.choices[0].delta.content
                    print(token, end="", flush=True)
                    full_text += token
        print()  # 换行
        return full_text
        
    except Exception:
        # 模拟流式输出效果
        text = "B站广告投放平台是面向广告主的一站式投放系统。平台支持开屏、信息流、Up主合作等多种广告形式。广告主可实时监控CTR、CVR、ROI等核心指标，智能优化投放策略。"
        print("流式输出（模拟）：", end="")
        for char in text:
            print(char, end="", flush=True)
            time.sleep(0.02)
        print()
        return text

stream_demo()

## 6. 异步并发调用

In [ ]:
import asyncio

async def concurrent_llm_calls():
    """并发调用多个 LLM 任务（适合 Agent 需要同时处理多个子任务）"""
    tasks = [
        "分析游戏广告的目标用户画像",
        "分析美妆广告的最佳投放时段",
        "分析教育广告的转化漏斗优化方向",
    ]
    
    print("并发执行 3 个分析任务...")
    start = time.time()
    
    # 并发执行
    results = await asyncio.gather(
        *[call_llm_async(
            prompt=task,
            system="你是广告分析专家，用一句话回答",
            max_tokens=100,
        ) for task in tasks],
        return_exceptions=True,
    )
    
    elapsed = time.time() - start
    print(f"完成（{elapsed:.1f}s），串行预计需要 {elapsed * 3:.1f}s")
    
    for task, result in zip(tasks, results):
        if isinstance(result, Exception):
            print(f"  ❌ {task[:20]}... → 失败: {result}")
        else:
            print(f"  ✅ {task[:20]}... → {str(result)[:60]}")

# Jupyter 中运行异步
await concurrent_llm_calls()

## 7. 成本追踪

In [ ]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class CostTracker:
    """LLM API 成本追踪器"""
    # 模型定价（美元/1M tokens）
    PRICING = {
        "gpt-4o-mini":   {"input": 0.15,  "output": 0.60},
        "gpt-4o":        {"input": 2.50,  "output": 10.0},
        "claude-haiku-4-5-20251001": {"input": 0.80,  "output": 4.0},
        "claude-sonnet-4-6": {"input": 3.0,  "output": 15.0},
        "qwen-turbo":    {"input": 0.056, "output": 0.17},
    }
    
    total_input_tokens: int = 0
    total_output_tokens: int = 0
    total_cost_usd: float = 0.0
    call_count: int = 0
    
    def track(self, model: str, input_tokens: int, output_tokens: int):
        pricing = self.PRICING.get(model, {"input": 1.0, "output": 3.0})
        cost = (input_tokens * pricing["input"] + output_tokens * pricing["output"]) / 1e6
        
        self.total_input_tokens += input_tokens
        self.total_output_tokens += output_tokens
        self.total_cost_usd += cost
        self.call_count += 1
        return cost
    
    def summary(self):
        print(f"\n=== API 调用成本汇总 ===")
        print(f"调用次数: {self.call_count}")
        print(f"总输入 tokens: {self.total_input_tokens:,}")
        print(f"总输出 tokens: {self.total_output_tokens:,}")
        print(f"总费用: ${self.total_cost_usd:.4f} (约 ¥{self.total_cost_usd * 7.2:.3f})")
        if self.call_count > 0:
            print(f"单次平均: ${self.total_cost_usd/self.call_count:.5f}")

# 使用示例
tracker = CostTracker()

# 模拟几次调用
tracker.track("gpt-4o-mini",   input_tokens=500,  output_tokens=150)
tracker.track("gpt-4o-mini",   input_tokens=1200, output_tokens=300)
tracker.track("claude-haiku-4-5-20251001",   input_tokens=800,  output_tokens=200)
tracker.track("qwen-turbo",    input_tokens=600,  output_tokens=100)

tracker.summary()

## 模型对比速查表

| 模型 | 上下文 | 输入价格 | 中文能力 | 速度 | 推荐场景 |
|------|--------|---------|---------|------|----------|
| GPT-4o-mini | 128K | $0.15/1M | ⭐⭐⭐ | 快 | 日常分析、分类、提取 |
| GPT-4o | 128K | $2.50/1M | ⭐⭐⭐⭐ | 中 | 复杂推理、代码生成 |
| Claude Haiku 4.5 | 200K | $0.80/1M | ⭐⭐⭐ | 极快 | 高并发批处理 |
| Claude Sonnet 4.6 | 200K | $3.00/1M | ⭐⭐⭐⭐⭐ | 中 | 长文档、复杂 Agent |
| Qwen-Turbo | 1M | ¥0.40/1M | ⭐⭐⭐⭐⭐ | 快 | 国内部署、中文任务 |
| Qwen-Max | 32K | ¥2.40/1M | ⭐⭐⭐⭐⭐ | 中 | 高精度中文任务 |

## 面试速记

| 问题 | 要点 |
|------|------|
| 如何降低 API 成本 | 用 mini 模型做分类/提取；缓存相同查询；减少 System Prompt 冗余 |
| 流式输出的应用场景 | 长文生成、聊天界面实时显示、用户体验改善 |
| 如何处理 API 限流 | 指数退避重试，`tenacity` 库或自定义 @retry 装饰器 |
| temperature 怎么设置 | 提取/分类用 0-0.3；文案创意用 0.7-1.0 |
| JSON 输出如何保证格式 | response_format={"type":"json_object"} + Pydantic 验证 |

**下一节**: `04_prompt_engineering.ipynb`